# 🚀 Huấn Luyện Mô Hình Phân Loại Hành Động - Contrastive Learning

Notebook này sử dụng **Contrastive Learning (SimCLR-style)** để học representations từ chuỗi keypoints, sau đó fine-tune cho phân loại.

## 📋 Tổng quan phương pháp
- **Phương pháp**: Two-Phase Training (Contrastive Pre-training + Fine-tuning)
- **Kiến trúc**: LSTM Encoder + Projection Head + Classifier Head
- **Phase 1 (60%)**: Pre-training với contrastive loss (NT-Xent)
- **Phase 2 (40%)**: Fine-tuning với classification loss
- **Đánh giá**: K-Fold Cross-Validation (5 folds)

## ✅ Ưu điểm
- Học được representations tổng quát và robust
- Tận dụng data augmentation tốt hơn
- Ít bị overfit nhờ self-supervised pre-training
- Có thể tận dụng unlabeled data (nếu có)
- Generalize tốt với dữ liệu mới

## ⚠️ Hạn chế
- Phức tạp hơn, khó debug
- Thời gian training lâu hơn (100 epochs)
- Cần tuning nhiều hyperparameters (temperature, augmentation, ...)
- Memory usage cao hơn (2 views mỗi sample)

## 🔄 So sánh với Classifier
| Tiêu chí | Classifier | Contrastive |
|----------|-----------|-------------|
| Độ phức tạp | Đơn giản ⭐ | Phức tạp ⭐⭐⭐ |
| Thời gian train | Nhanh (50 epochs) | Lâu (100 epochs) |
| Độ chính xác | Tốt | Có thể tốt hơn |
| Generalization | Trung bình | Tốt |
| Unlabeled data | Không dùng được | Có thể dùng |

## 1. Import Libraries

Các thư viện cần thiết cho contrastive learning:
- **PyTorch**: Deep learning framework
- **scikit-learn**: K-Fold CV, metrics
- **matplotlib**: Visualization
- **tqdm**: Progress bars
- **random**: Data augmentation

In [18]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from tqdm import tqdm
import random

from IPython.display import clear_output

# 📊 Configure tqdm để output gọn gàng hơn trong notebooktqdm.pandas()

## 2. Cấu hình Training

Thiết lập các hyperparameters cho contrastive learning:
- **NUM_EPOCHS**: 100 (60 pretrain + 40 finetune)
- **TEMPERATURE**: 0.5 - hyperparameter quan trọng cho contrastive loss
- **Augmentation**: Gaussian noise (σ=0.02), Random scaling (0.9-1.1x)

In [ ]:
DATASET_DIR = "../dataset"  # Relative to Behavier_recognition folder
SEQUENCE_LENGTH = 30  # Number of consecutive valid frames
STRIDE = 15  # Sliding window stride
MIN_CONFIDENCE = 0.3  # Minimum keypoint confidence
MIN_VALID_KEYPOINTS = 8  # Minimum valid keypoints per frame (out of 12)
BATCH_SIZE = 48  # Increased for faster training (contrastive needs 2 views)
NUM_EPOCHS = 100  # Contrastive learning needs more epochs
LEARNING_RATE = 0.001
TEMPERATURE = 0.5  # Temperature for contrastive loss
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.3
K_FOLDS = 5  # Number of folds for K-fold cross-validation
NUM_WORKERS = 4  # Parallel data loading

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():

    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Batch Size: {BATCH_SIZE} | Num Workers: {NUM_WORKERS}")

    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"K-Fold CV: {K_FOLDS} folds")

K-Fold CV: 5 folds
Using device: cuda


## 3. Contrastive Dataset với Data Augmentation

### 🔍 Tính năng chính:
- **12 body keypoints**: Vai (5,6), Tay (7,8,9,10), Hông (11,12), Chân (13,14,15,16)
- **Quality filtering**: Giống classifier (min_confidence=0.3, min_valid_keypoints=8)
- **Data augmentation** cho contrastive learning:
  - ✅ **Gaussian noise**: σ=0.02 → tăng robustness với noise keypoints
  - ✅ **Random scaling**: 0.9-1.1x → mô phỏng khoảng cách khác nhau
  - ⚠️ **Temporal jitter**: Đã loại bỏ để đảm bảo sequence length cố định

### 💡 Tại sao cần augmentation?
- Tạo **2 views khác nhau** của cùng 1 sequence
- Model học được **invariance** với các biến đổi nhỏ
- Tăng **diversity** của training data
- Cải thiện **generalization** ability

### 🎯 Contrastive Learning principle:
```
Cùng 1 sequence → 2 views (v1, v2)
→ Model phải học: v1 và v2 gần nhau (positive pair)
→ v1 và sequences khác xa nhau (negative pairs)
```

In [20]:
class ContrastiveDataset(Dataset):
    """Dataset for contrastive learning with data augmentation"""
    
    BODY_KEYPOINTS = [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
    
    def __init__(self, data_paths, labels, sequence_length=30, stride=15,
                 min_confidence=0.3, min_valid_keypoints=8, augment=True):
        self.sequences = []
        self.labels = []
        self.sequence_length = sequence_length
        self.stride = stride
        self.min_confidence = min_confidence
        self.min_valid_keypoints = min_valid_keypoints
        self.augment = augment
        
        print(f"\nLoading contrastive dataset:")
        print(f"  - Min confidence: {min_confidence}")
        print(f"  - Min valid keypoints: {min_valid_keypoints}/{len(self.BODY_KEYPOINTS)}")
        print(f"  - Augmentation: {augment}")
        
        for path, label in zip(data_paths, labels):
            self._load_json(path, label)
        
        print(f"\nTotal sequences: {len(self.sequences)}")
    
    def _is_frame_valid(self, frame):
        if not isinstance(frame, list) or len(frame) == 0:
            return False
        valid_count = 0
        for kp_idx in self.BODY_KEYPOINTS:
            if kp_idx < len(frame):
                keypoint = frame[kp_idx]
                if len(keypoint) >= 3 and keypoint[2] >= self.min_confidence:
                    valid_count += 1
        return valid_count >= self.min_valid_keypoints
    
    def _find_valid_sequences(self, frames):
        valid_sequences = []
        current_sequence = []
        for i, frame in enumerate(frames):
            if self._is_frame_valid(frame):
                current_sequence.append(i)
                if len(current_sequence) >= self.sequence_length:
                    start_idx = len(current_sequence) - self.sequence_length
                    if start_idx % self.stride == 0 or start_idx == 0:
                        valid_sequences.append(current_sequence[-self.sequence_length:])
            else:
                current_sequence = []
        return valid_sequences
    
    def _load_json(self, path, label):
        try:
            with open(path, 'r') as f:
                data = json.load(f)
            frames = data.get('frames', [])
            if len(frames) < self.sequence_length:
                return
            valid_sequences = self._find_valid_sequences(frames)
            if len(valid_sequences) == 0:
                return
            for seq_indices in valid_sequences:
                sequence_frames = [frames[i] for i in seq_indices]
                keypoints = self._extract_keypoints(sequence_frames)
                if keypoints is not None:
                    self.sequences.append(keypoints)
                    self.labels.append(label)
        except Exception as e:
            print(f"❌ Error loading {path}: {e}")
    
    def _extract_keypoints(self, sequence):
        keypoints_seq = []
        for frame in sequence:
            frame_kps = []
            for kp_idx in self.BODY_KEYPOINTS:
                if kp_idx < len(frame):
                    keypoint = frame[kp_idx]
                    if len(keypoint) >= 3:
                        x = np.clip(keypoint[0], 0, 224)
                        y = np.clip(keypoint[1], 0, 224)
                        conf = np.clip(keypoint[2], 0, 1)
                        frame_kps.extend([x, y, conf])
                    else:
                        frame_kps.extend([0, 0, 0])
                else:
                    frame_kps.extend([0, 0, 0])
            keypoints_seq.append(frame_kps)
        
        keypoints_array = np.array(keypoints_seq, dtype=np.float32)
        keypoints_array[:, 0::3] /= 224.0
        keypoints_array[:, 1::3] /= 224.0
        return keypoints_array
    
    def _augment_sequence(self, seq):
        """Apply data augmentation while maintaining sequence length"""
        seq = seq.copy()
        
        # Add Gaussian noise
        if random.random() > 0.5:
            noise = np.random.normal(0, 0.02, seq.shape)
            seq = seq + noise
        
        # Random scaling
        if random.random() > 0.5:
            scale = np.random.uniform(0.9, 1.1)
            seq[:, 0::3] *= scale  # x coords
            seq[:, 1::3] *= scale  # y coords
        
        # Clip to valid range
        seq[:, 0::3] = np.clip(seq[:, 0::3], 0, 1)
        seq[:, 1::3] = np.clip(seq[:, 1::3], 0, 1)
        seq[:, 2::3] = np.clip(seq[:, 2::3], 0, 1)
        
        return seq
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        sequence = self.sequences[idx]
        label = self.labels[idx]
        
        if self.augment:
            # Create two augmented views with same length
            view1 = self._augment_sequence(sequence)
            view2 = self._augment_sequence(sequence)
            # Both views now have same length as original
            return torch.FloatTensor(view1), torch.FloatTensor(view2), torch.LongTensor([label])[0]
        else:
            return torch.FloatTensor(sequence), torch.LongTensor([label])[0]

## 4. Kiến trúc Model - Three-Component Architecture

### 🏗️ Cấu trúc 3 thành phần:

#### 1️⃣ **LSTMEncoder** (Feature Extractor)
```
Input (30, 36) → Bi-LSTM (2 layers, 128 hidden) → Embedding (256)
```
- Shared encoder cho cả 2 phases
- Extract temporal features từ keypoint sequences

#### 2️⃣ **Projection Head** (Phase 1 - Pre-training)
```
Embedding (256) → FC (128) + ReLU + Dropout → Projection (128) → L2 Normalize
```
- Chỉ dùng trong phase 1 (contrastive learning)
- Project embeddings vào space phù hợp cho contrastive loss
- L2 normalize để tính cosine similarity

#### 3️⃣ **Classification Head** (Phase 2 - Fine-tuning)
```
Embedding (256) → FC (128) + ReLU + Dropout → Output (2)
```
- Thay thế projection head trong phase 2
- Fine-tune encoder + classifier cho task phân loại

### 🎓 Ưu điểm của kiến trúc:
- **Encoder** học được representations tổng quát từ contrastive learning
- **Projection head** tách biệt pre-training và fine-tuning
- **Classification head** chỉ học trên representations tốt

In [21]:
class LSTMEncoder(nn.Module):
    """Bi-directional LSTM encoder"""
    
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
        super(LSTMEncoder, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        self.embedding_size = hidden_size * 2
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        return lstm_out[:, -1, :]  # Take last timestep


class ContrastiveModel(nn.Module):
    """Contrastive learning model with projection head"""
    
    def __init__(self, input_size, hidden_size=128, num_layers=2, 
                 projection_dim=128, dropout=0.3):
        super(ContrastiveModel, self).__init__()
        self.encoder = LSTMEncoder(input_size, hidden_size, num_layers, dropout)
        
        # Projection head for contrastive learning
        self.projection = nn.Sequential(
            nn.Linear(self.encoder.embedding_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, projection_dim)
        )
    
    def forward(self, x):
        embeddings = self.encoder(x)
        projections = self.projection(embeddings)
        return F.normalize(projections, dim=1)  # L2 normalize


class ContrastiveClassifier(nn.Module):
    """Add classification head to pretrained encoder"""
    
    def __init__(self, encoder, num_classes=2, dropout=0.3):
        super(ContrastiveClassifier, self).__init__()
        self.encoder = encoder
        hidden_size = encoder.embedding_size
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_classes)
        )
    
    def forward(self, x):
        embeddings = self.encoder(x)
        return self.classifier(embeddings)

## 5. Contrastive Loss - NT-Xent (SimCLR)

### 📐 Normalized Temperature-scaled Cross Entropy Loss

#### 🔢 Công thức:
Với batch size N, có 2N samples (2 views):
```
ℓ(i) = -log[ exp(sim(zi, zj)/τ) / Σ exp(sim(zi, zk)/τ) ]
```
Trong đó:
- `zi, zj`: projections của 2 views từ cùng 1 sample (positive pair)
- `zk`: projections của các samples khác (negative pairs)
- `sim(·,·)`: cosine similarity
- `τ`: temperature (0.5) - điều chỉnh độ "mạnh" của loss

#### 💡 Intuition:
- **Positive pairs** (v1, v2 của cùng sequence): đẩy GẦN nhau
- **Negative pairs** (v1 với sequences khác): đẩy XA nhau
- **Temperature**: τ nhỏ → phân biệt hard negatives tốt hơn

#### 🎯 Mục tiêu:
Model học được embeddings mà:
- Cùng action (walking) → embeddings gần nhau
- Khác action (non-walking) → embeddings xa nhau
- Robust với augmentation

In [22]:
def nt_xent_loss(z1, z2, temperature=0.5):
    """
    Normalized Temperature-scaled Cross Entropy Loss (NT-Xent) from SimCLR
    
    Args:
        z1, z2: Normalized projections from two augmented views [batch_size, projection_dim]
        temperature: Temperature parameter for scaling
    """
    batch_size = z1.shape[0]
    
    # Concatenate representations
    z = torch.cat([z1, z2], dim=0)  # [2*batch_size, projection_dim]
    
    # Compute similarity matrix
    sim_matrix = torch.mm(z, z.t()) / temperature  # [2*batch_size, 2*batch_size]
    
    # Create mask for positive pairs
    # Diagonal: self-similarity (exclude)
    # Off-diagonal: positive pairs at (i, i+batch_size) and (i+batch_size, i)
    mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z.device)
    sim_matrix = sim_matrix.masked_fill(mask, -9e15)
    
    # Positive pairs: (i, i+batch_size) for first half, (i, i-batch_size) for second half
    pos_sim = torch.cat([
        sim_matrix[:batch_size, batch_size:].diag(),
        sim_matrix[batch_size:, :batch_size].diag()
    ])
    
    # Compute loss: -log(exp(pos) / sum(exp(all)))
    loss = -pos_sim + torch.logsumexp(sim_matrix, dim=1)
    loss = loss.mean()
    
    return loss

## 6. Load and Prepare Dataset

In [23]:
# Load all dataset paths
walking_dir = os.path.join(DATASET_DIR, 'walking')
non_walking_dir = os.path.join(DATASET_DIR, 'non-walking')

data_paths = []
labels = []

if os.path.exists(walking_dir):
    for filename in os.listdir(walking_dir):
        if filename.endswith('.json'):
            data_paths.append(os.path.join(walking_dir, filename))
            labels.append(1)

if os.path.exists(non_walking_dir):
    for filename in os.listdir(non_walking_dir):
        if filename.endswith('.json'):
            data_paths.append(os.path.join(non_walking_dir, filename))
            labels.append(0)

print(f"Total files: {len(data_paths)}")
print(f"  - Walking: {sum(labels)}")
print(f"  - Non-walking: {len(labels) - sum(labels)}")

data_paths = np.array(data_paths)
labels = np.array(labels)

Total files: 2656
  - Walking: 2553
  - Non-walking: 103


## 7. Two-Phase Training với K-Fold Cross-Validation

### 🔄 K-Fold CV (5 folds) + Two-Phase Training

#### 📊 Tổng quan:
- **Total epochs**: 100 (60 pretrain + 40 finetune)
- **K-Folds**: 5 folds, stratified split
- **Output**: 5 models (1 model per fold)

---

### 🎯 Phase 1: Contrastive Pre-training (60 epochs)

#### Mục tiêu:
- Học representations tổng quát từ augmented views
- Không cần labels (self-supervised)
- Maximize agreement giữa 2 views của cùng 1 sequence

#### Training setup:
- **Dataset**: ContrastiveDataset với `augment=True`
- **Loss**: NT-Xent contrastive loss
- **Optimizer**: Adam (lr=0.001)
- **Output**: Pre-trained encoder

#### 💡 Tại sao pre-training?
- Encoder học được features tổng quát
- Tránh overfitting trên labeled data
- Cải thiện generalization

---

### 🎓 Phase 2: Fine-tuning Classification (40 epochs)

#### Mục tiêu:
- Adapt pre-trained encoder cho task phân loại
- Fine-tune cả encoder + classifier head

#### Training setup:
- **Dataset**: ContrastiveDataset với `augment=False`
- **Loss**: CrossEntropyLoss
- **Optimizer**: Adam (lr=0.0001) ← lr nhỏ hơn 10x để giữ pre-trained weights
- **Scheduler**: ReduceLROnPlateau
- **Early stopping**: Save best model theo validation accuracy

#### 💾 Output mỗi fold:
- Model: `action_contrastive_fold{N}.pth`
- Chứa: encoder + classifier head
- Best model theo validation accuracy

---

### 📈 Ưu điểm của Two-Phase:
1. **Phase 1**: Learn general representations (self-supervised)
2. **Phase 2**: Specialize for classification (supervised)
3. Kết hợp ưu điểm của cả 2 approaches

In [ ]:
# K-Fold Cross Validation
kfold = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

fold_results = []
all_histories = []

PRETRAIN_EPOCHS = int(NUM_EPOCHS * 0.6)
FINETUNE_EPOCHS = NUM_EPOCHS - PRETRAIN_EPOCHS

print("\n" + "="*60)
print(f"K-FOLD CV with TWO-PHASE TRAINING ({K_FOLDS} folds)")
print("="*60)
print(f"Phase 1 (Contrastive): {PRETRAIN_EPOCHS} epochs")
print(f"Phase 2 (Classification): {FINETUNE_EPOCHS} epochs")

for fold, (train_idx, val_idx) in enumerate(kfold.split(data_paths)):
    print(f"\n{'='*60}")
    print(f"FOLD {fold + 1}/{K_FOLDS}")
    print(f"{'='*60}")
    
    train_paths_fold = data_paths[train_idx]
    train_labels_fold = labels[train_idx]
    val_paths_fold = data_paths[val_idx]
    val_labels_fold = labels[val_idx]
    
    print(f"\nFold {fold + 1} split:")
    print(f"  Train files: {len(train_paths_fold)}")
    print(f"  Val files: {len(val_paths_fold)}")
    
    # Phase 1: Contrastive Pre-training
    print(f"\n{'='*60}")
    print(f"PHASE 1: CONTRASTIVE PRE-TRAINING")
    print(f"{'='*60}")
    
    train_dataset_contrast = ContrastiveDataset(
        train_paths_fold, train_labels_fold,
        SEQUENCE_LENGTH, STRIDE,
        MIN_CONFIDENCE, MIN_VALID_KEYPOINTS,
        augment=True
    )
    
    if len(train_dataset_contrast) == 0:
        print(f"⚠️ Fold {fold + 1}: No valid sequences, skipping...")
        continue
    
    train_loader_contrast = DataLoader(
        train_dataset_contrast, 
        batch_size=BATCH_SIZE,
        shuffle=True, 
        num_workers=NUM_WORKERS,
        pin_memory=True if torch.cuda.is_available() else False,
        persistent_workers=True if NUM_WORKERS > 0 else False
    )
    
    # Get input size
    view1, view2, _ = train_dataset_contrast[0]
    input_size = view1.shape[1]
    
    # Create contrastive model
    contrastive_model = ContrastiveModel(
        input_size=input_size,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        projection_dim=HIDDEN_SIZE,
        dropout=DROPOUT
    ).to(device)
    
    optimizer_contrast = optim.Adam(contrastive_model.parameters(), lr=LEARNING_RATE)
    
    contrastive_losses = []
    
    for epoch in range(PRETRAIN_EPOCHS):
        contrastive_model.train()
        epoch_loss = 0.0
        
        for view1, view2, _ in tqdm(train_loader_contrast, desc=f"Pretrain Epoch {epoch+1}/{PRETRAIN_EPOCHS}"):
            view1 = view1.to(device)
            view2 = view2.to(device)
            
            optimizer_contrast.zero_grad(set_to_none=True)
            z1 = contrastive_model(view1)
            z2 = contrastive_model(view2)
            loss = nt_xent_loss(z1, z2, TEMPERATURE)
            loss.backward()
            optimizer_contrast.step()
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(train_loader_contrast)
        contrastive_losses.append(avg_loss)
        
        if (epoch + 1) % 10 == 0:
            # Clear output mỗi 20 epochs để notebook gọn hơn
            if (epoch + 1) % 20 == 0:
                clear_output(wait=True)
                print(f"Fold {fold + 1}/{K_FOLDS} - Phase 1: Pretrain Progress...")
            print(f"Epoch {epoch+1}: Contrastive Loss={avg_loss:.4f}")
    
    # Phase 2: Fine-tuning with Classification
    print(f"\n{'='*60}")
    print(f"PHASE 2: FINE-TUNING WITH CLASSIFICATION")
    print(f"{'='*60}")
    
    train_dataset_finetune = ContrastiveDataset(
        train_paths_fold, train_labels_fold,
        SEQUENCE_LENGTH, STRIDE,
        MIN_CONFIDENCE, MIN_VALID_KEYPOINTS,
        augment=False
    )
    
    val_dataset = ContrastiveDataset(
        val_paths_fold, val_labels_fold,
        SEQUENCE_LENGTH, STRIDE,
        MIN_CONFIDENCE, MIN_VALID_KEYPOINTS,
        augment=False
    )
    
    if len(val_dataset) == 0:
        print(f"⚠️ Fold {fold + 1}: No validation sequences, skipping...")
        continue
    
    train_loader_finetune = DataLoader(
        train_dataset_finetune, 
        batch_size=BATCH_SIZE,
        shuffle=True, 
        num_workers=NUM_WORKERS,
        pin_memory=True if torch.cuda.is_available() else False,
        persistent_workers=True if NUM_WORKERS > 0 else False
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=BATCH_SIZE * 2,  # Larger batch for validation
        shuffle=False, 
        num_workers=NUM_WORKERS,
        pin_memory=True if torch.cuda.is_available() else False,
        persistent_workers=True if NUM_WORKERS > 0 else False
    )
    
    # Create classifier with pretrained encoder
    classifier = ContrastiveClassifier(
        contrastive_model.encoder,
        num_classes=2,
        dropout=DROPOUT
    ).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer_finetune = optim.Adam(classifier.parameters(), lr=LEARNING_RATE * 0.1)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer_finetune, mode='max', factor=0.5, patience=5)
    
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    best_val_acc = 0.0
    start_finetune_epoch = 0
    
    # 🔄 Load checkpoint if exists (để tiếp tục training)
    checkpoint_path = f'action_contrastive_fold{fold+1}.pth'
    if os.path.exists(checkpoint_path):
        print(f"\n📂 Found checkpoint: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path)
        
        # Nếu checkpoint chỉ chứa state_dict
        if isinstance(checkpoint, dict) and 'model_state_dict' not in checkpoint:
            classifier.load_state_dict(checkpoint)
            print(f"✅ Loaded model weights from checkpoint")
        # Nếu checkpoint chứa đầy đủ thông tin
        elif isinstance(checkpoint, dict):
            classifier.load_state_dict(checkpoint['model_state_dict'])
            optimizer_finetune.load_state_dict(checkpoint['optimizer_state_dict'])
            start_finetune_epoch = checkpoint.get('epoch', 0)
            best_val_acc = checkpoint.get('best_val_acc', 0.0)
            train_losses = checkpoint.get('train_losses', [])
            val_losses = checkpoint.get('val_losses', [])
            train_accs = checkpoint.get('train_accs', [])
            val_accs = checkpoint.get('val_accs', [])
            contrastive_losses = checkpoint.get('contrastive_losses', contrastive_losses)
            print(f"✅ Loaded full checkpoint (epoch {start_finetune_epoch}, best_val_acc={best_val_acc:.4f})")
            print(f"   Resuming from epoch {start_finetune_epoch + 1}")
    else:
        print(f"\n🆕 No checkpoint found. Training from scratch.")
    
    for epoch in range(start_finetune_epoch, FINETUNE_EPOCHS):
        # Training
        classifier.train()
        train_loss = 0.0
        train_preds = []
        train_labels_list = []
        
        # Disable tqdm for faster training (show every 10 epochs)
        show_progress = (epoch + 1) % 10 == 0 or epoch == 0
        loader_iter = tqdm(train_loader_finetune, desc=f"Finetune Epoch {epoch+1}/{FINETUNE_EPOCHS}") if show_progress else train_loader_finetune
        
        for sequences, batch_labels in loader_iter:
            sequences = sequences.to(device, non_blocking=True)
            batch_labels = batch_labels.to(device, non_blocking=True)
            
            optimizer_finetune.zero_grad(set_to_none=True)
            outputs = classifier(sequences)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer_finetune.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_preds.extend(predicted.cpu().numpy())
            train_labels_list.extend(batch_labels.cpu().numpy())
        
        train_loss /= len(train_loader_finetune)
        train_acc = accuracy_score(train_labels_list, train_preds)
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        
        # Validation
        classifier.eval()
        val_loss = 0.0
        val_preds = []
        val_labels_list = []
        
        with torch.no_grad():
            for sequences, batch_labels in val_loader:
                sequences = sequences.to(device, non_blocking=True)
                batch_labels = batch_labels.to(device, non_blocking=True)
                
                outputs = classifier(sequences)
                loss = criterion(outputs, batch_labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_preds.extend(predicted.cpu().numpy())
                val_labels_list.extend(batch_labels.cpu().numpy())
        
        val_loss /= len(val_loader)
        val_acc = accuracy_score(val_labels_list, val_preds)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        scheduler.step(val_acc)

        # Clear output mỗi 10 epochs để notebook gọn hơn
        if (epoch + 1) % 10 == 0:
            clear_output(wait=True)
            print(f"Fold {fold + 1}/{K_FOLDS} - Phase 2: Finetune Progress...")
        print(f"Epoch {epoch+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}, LR={optimizer_finetune.param_groups[0]['lr']:.6f}")
    
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            # 💾 Save full checkpoint (model + optimizer + training state)
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': classifier.state_dict(),
                'optimizer_state_dict': optimizer_finetune.state_dict(),
                'best_val_acc': best_val_acc,
                'train_losses': train_losses,
                'val_losses': val_losses,
                'train_accs': train_accs,
                'val_accs': val_accs,
                'contrastive_losses': contrastive_losses
            }
            torch.save(checkpoint, f'action_contrastive_fold{fold+1}.pth')
            if (epoch + 1) % 10 == 0:
                print(f"  ✓ Saved best model (Val Acc: {best_val_acc:.4f})")
    
    # Store results
    fold_results.append({
        'fold': fold + 1,
        'best_val_acc': best_val_acc,
        'final_train_acc': train_accs[-1],
        'final_val_acc': val_accs[-1],
        'val_preds': val_preds,
        'val_labels': val_labels_list
    })
    
    all_histories.append({
        'contrastive_losses': contrastive_losses,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accs': train_accs,
        'val_accs': val_accs
    })
    
    print(f"\nFold {fold + 1} Results:")
    print(f"  Best Val Accuracy: {best_val_acc:.4f}")
    print(f"  Final Train Accuracy: {train_accs[-1]:.4f}")
    print(f"  Final Val Accuracy: {val_accs[-1]:.4f}")
    
    print(f"\nFold {fold + 1} Classification Report:")
    print(classification_report(val_labels_list, val_preds,
                                target_names=['Non-Walking', 'Walking'],
                                zero_division=0))

print("\n" + "="*60)
print("K-FOLD CROSS-VALIDATION COMPLETE")
print("="*60)

Fold 2/5 - Phase 1: Pretrain Progress...
Epoch 20: Contrastive Loss=2.5540


Pretrain Epoch 26/60:  33%|███▎      | 172/525 [00:01<00:03, 92.05it/s]



KeyboardInterrupt: 

## 8. Tổng hợp kết quả và Visualization

### 📊 Metrics được tính:
- **Best Validation Accuracy**: Độ chính xác tốt nhất của mỗi fold
- **Mean ± Std**: Trung bình và độ lệch chuẩn qua 5 folds
- **Per-fold results**: Kết quả chi tiết từng fold

### 📈 Các biểu đồ:
1. **Accuracy per fold**: So sánh độ chính xác giữa các folds
2. **Contrastive loss**: Loss của phase 1 (pre-training)
3. **Training loss**: Loss của phase 2 (fine-tuning)
4. **Validation accuracy**: Accuracy của phase 2
5. **Distribution**: Phân bố accuracy qua các folds
6. **Summary**: Tổng hợp kết quả text

### 🔍 Confusion Matrix:
- Tổng hợp predictions từ tất cả 5 folds
- Đánh giá chi tiết: True Positives, False Positives, etc.
- Classification report: Precision, Recall, F1-score

In [ ]:
# Calculate average metrics
avg_best_val_acc = np.mean([r['best_val_acc'] for r in fold_results])
avg_final_train_acc = np.mean([r['final_train_acc'] for r in fold_results])
avg_final_val_acc = np.mean([r['final_val_acc'] for r in fold_results])

std_best_val_acc = np.std([r['best_val_acc'] for r in fold_results])
std_final_val_acc = np.std([r['final_val_acc'] for r in fold_results])

print("\n" + "="*60)
print("AVERAGE RESULTS ACROSS ALL FOLDS (Contrastive Learning)")
print("="*60)
print(f"\nBest Validation Accuracy:")
print(f"  Mean: {avg_best_val_acc:.4f} ± {std_best_val_acc:.4f}")
print(f"\nFinal Training Accuracy:")
print(f"  Mean: {avg_final_train_acc:.4f}")
print(f"\nFinal Validation Accuracy:")
print(f"  Mean: {avg_final_val_acc:.4f} ± {std_final_val_acc:.4f}")

print(f"\n\nPer-fold results:")
for r in fold_results:
    print(f"  Fold {r['fold']}: Best Val Acc = {r['best_val_acc']:.4f}")

# Select and save best model as final checkpoint
best_fold_idx = np.argmax([r['best_val_acc'] for r in fold_results])
best_fold = fold_results[best_fold_idx]
best_model_path = f'action_contrastive_fold{best_fold["fold"]}.pth'

print(f"\n{'='*60}")
print(f"BEST MODEL SELECTION")
print(f"{'='*60}")
print(f"Best performing fold: Fold {best_fold['fold']}")
print(f"Best validation accuracy: {best_fold['best_val_acc']:.4f}")
print(f"Source: {best_model_path}")

# Load checkpoint và chỉ lưu model_state_dict cho inference
import shutil
if os.path.exists(best_model_path):
    checkpoint = torch.load(best_model_path)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model_weights = checkpoint['model_state_dict']
    else:
        model_weights = checkpoint
    
    final_checkpoint_path = 'action_contrastive_best.pth'
    torch.save(model_weights, final_checkpoint_path)
    
    print(f"✅ Saved best model to: {final_checkpoint_path}")
    print(f"   (Model weights only, optimized for inference)")
    print(f"\n💡 To use this model for inference:")
    print(f"   encoder = LSTMEncoder(input_size=36, hidden_size=128, num_layers=2)")
    print(f"   classifier = ContrastiveClassifier(encoder, num_classes=2, dropout=0.3)")
    print(f"   classifier.load_state_dict(torch.load('{final_checkpoint_path}'))")
    print(f"   classifier.eval()")
else:
    print(f"⚠️ Best model file not found: {best_model_path}")

In [ ]:
# Visualize results
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

fold_nums = [r['fold'] for r in fold_results]
best_accs = [r['best_val_acc'] for r in fold_results]

# Plot 1: Per-fold accuracy
axes[0, 0].bar(fold_nums, best_accs, color='steelblue', alpha=0.7)
axes[0, 0].axhline(y=avg_best_val_acc, color='red', linestyle='--', label=f'Mean: {avg_best_val_acc:.4f}')
axes[0, 0].set_xlabel('Fold')
axes[0, 0].set_ylabel('Best Validation Accuracy')
axes[0, 0].set_title('Contrastive Learning: Accuracy per Fold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Contrastive loss curves
for i, h in enumerate(all_histories):
    axes[0, 1].plot(h['contrastive_losses'], alpha=0.5, label=f"Fold {i+1}")
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Contrastive Loss')
axes[0, 1].set_title('Phase 1: Contrastive Pre-training Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Fine-tuning train loss
for i, h in enumerate(all_histories):
    axes[0, 2].plot(h['train_losses'], alpha=0.5, label=f"Fold {i+1}")
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('Training Loss')
axes[0, 2].set_title('Phase 2: Fine-tuning Training Loss')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Plot 4: Validation accuracy
for i, h in enumerate(all_histories):
    axes[1, 0].plot(h['val_accs'], alpha=0.5, label=f"Fold {i+1}")
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Validation Accuracy')
axes[1, 0].set_title('Phase 2: Validation Accuracy')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 5: Accuracy distribution
axes[1, 1].boxplot([best_accs], labels=['Best Val Acc'])
axes[1, 1].axhline(y=avg_best_val_acc, color='red', linestyle='--', alpha=0.7)
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].set_title('Accuracy Distribution')
axes[1, 1].grid(True, alpha=0.3)

# Plot 6: Comparison text
axes[1, 2].axis('off')
summary_text = f"""
Contrastive Learning Results
{'='*35}

K-Fold CV: {K_FOLDS} folds
Pre-training: {PRETRAIN_EPOCHS} epochs
Fine-tuning: {FINETUNE_EPOCHS} epochs

Mean Accuracy:
{avg_best_val_acc:.4f} ± {std_best_val_acc:.4f}

Best Fold: {fold_results[np.argmax(best_accs)]['fold']}
Best Accuracy: {max(best_accs):.4f}

Models saved:
action_contrastive_fold1.pth
... through fold{K_FOLDS}.pth
"""
axes[1, 2].text(0.1, 0.5, summary_text, fontsize=11, family='monospace', verticalalignment='center')

plt.tight_layout()
plt.savefig('contrastive_kfold_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nVisualization saved to 'contrastive_kfold_results.png'")

In [ ]:
# Aggregate confusion matrix
all_true = []
all_pred = []

for r in fold_results:
    all_true.extend(r['val_labels'])
    all_pred.extend(r['val_preds'])

cm = confusion_matrix(all_true, all_pred)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)

class_names = ['Non-walking', 'Walking']
ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=class_names, yticklabels=class_names,
       title='Contrastive Learning: Confusion Matrix (All Folds)',
       ylabel='True label',
       xlabel='Predicted label')

plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=16)

plt.tight_layout()
plt.savefig('contrastive_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nOverall Classification Report (Contrastive Learning):")
print(classification_report(all_true, all_pred, target_names=class_names, digits=4))

print("\n" + "="*60)
print("CONTRASTIVE LEARNING K-FOLD CV COMPLETE!")
print("="*60)
print(f"\nFinal Result: {avg_best_val_acc:.4f} ± {std_best_val_acc:.4f}")
print(f"\nBest performing fold: Fold {fold_results[np.argmax(best_accs)]['fold']}")
print(f"  Accuracy: {max(best_accs):.4f}")